In [1]:
import copy
import json
import numpy as np
import os
import pandas as pd
from scipy.stats import zscore
from statsmodels.stats.inter_rater import fleiss_kappa
from scipy.stats import ranksums

SYSTEMS_PATH = 'data/correlation/en/'
REFERENCES_PATH = 'data/human_evaluation/data_REF.json'

In [10]:
# READ FILES
rdfs = json.load(open(REFERENCES_PATH))
sys_files = [w for w in os.listdir(SYSTEMS_PATH) if not w.startswith('.')]

doc_id = 1
data = []
for sys_file in sys_files:
    results = json.load(open(os.path.join(SYSTEMS_PATH, sys_file)))
    submission_id = sys_file.replace('.json', '').replace('human_eval_id_', '')

    for sample_id in results:
        entry = [w for w in rdfs['entries'] if list(w.keys())[0] == sample_id][0]
        hypothesis = results[sample_id]['text']
        for worker_id in results[sample_id]['hits']:
            assign = results[sample_id]['hits'][worker_id]
            inp = {
                'id': doc_id,
                'sample_id': sample_id,
                'submission_id': submission_id,
                'worker_id': worker_id,
                'category': entry[sample_id]['category'],
                'size': entry[sample_id]['size'],
                'hyp': hypothesis,
                'refs': [w['lex'] for w in entry[sample_id]['lexicalisations']]
            }
            inp.update(assign)
            data.append(inp)
            doc_id += 1

In [12]:
# NORMALIZE HUMAN SCORES WITH THEIR Z-SCORES

normdata = []
worker_ids = set([w['worker_id'] for w in data])
for worker_id in worker_ids:
    fdata = [w for w in data if w['worker_id'] == worker_id]
    
    ids = [w['id'] for w in fdata]
    correctness = zscore([w['Correctness'] for w in fdata])
    coverage = zscore([w['DataCoverage'] for w in fdata])
    fluency = zscore([w['Fluency'] for w in fdata])
    relevance = zscore([w['Relevance'] for w in fdata])
    structure = zscore([w['TextStructure'] for w in fdata])
    
    for i, id_ in enumerate(ids):
        for j, row in enumerate(data):
            if row['id'] == id_:
                row_ = copy.copy(row)
                row_['Correctness'] = correctness[i]
                row_['DataCoverage'] = coverage[i]
                row_['Fluency'] = fluency[i]
                row_['Relevance'] = relevance[i]
                row_['TextStructure'] = structure[i]
                normdata.append(row_)
                break

/usr/local/lib/python3.7/site-packages/scipy/stats/stats.py:2315: RuntimeWarning: invalid value encountered in true_divide
  return (a - mns) / sstd


In [20]:
# averaging the human evaluation Z-scores per trial
submission_ids = sorted(list(set([w['submission_id'] for w in normdata])))
sample_ids = sorted(list(set([w['sample_id'] for w in normdata])), key=lambda x: int(x))

finaldata = []
for submission_id in submission_ids:
    for sample_id in sample_ids:
        fdata = [w for w in normdata if w['submission_id'] == submission_id and w['sample_id'] == sample_id]

        if len(fdata) > 0:
            assert len(set([w['hyp'] for w in fdata])) == 1
            finaldata.append({
                'submission_id': submission_id,
                'size': fdata[0]['size'],
                'sample_id': sample_id,
                'category': fdata[0]['category'],
                'hyp': fdata[0]['hyp'],
                'refs': fdata[0]['refs'],
                'Correctness': np.nan_to_num(np.mean(np.nan_to_num([w['Correctness'] for w in fdata]))),
                'DataCoverage': np.nan_to_num(np.mean(np.nan_to_num([w['DataCoverage'] for w in fdata]))),
                'Fluency': np.nan_to_num(np.mean(np.nan_to_num([w['Fluency'] for w in fdata]))),
                'Relevance': np.nan_to_num(np.mean(np.nan_to_num([w['Relevance'] for w in fdata]))),
                'TextStructure': np.nan_to_num(np.mean(np.nan_to_num([w['TextStructure'] for w in fdata])))
            })

In [22]:
import eval



ModuleNotFoundError: No module named 'bert_score'